# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5. 
     

No, 100 successful demonstrations only provide evidence of competence with the tested conditions. But different environments, mug types, fill levels, obstacles, or maybe the human movement may produce some failure. General competence requires systematic testing across varied conditions and clearly defined limits.

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
     
During a turn the coffee may spill because the acceleration causes the liquid to move toward the rim. The original run did not test the assumption that the same turning speed and trajectory would remain safe at higher fill levles. The robot should reduce its speed or adjust the trajectory.

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
     
     The robot should ask Eva whether the white mug is acceptable before proceeding
     The decisive evidence is Evas explicit confirmation that she accepts the white mug. Without its confirmation the grounding criterion of the warrant of the warrant gate is not satisfied. 

4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.

   T1: Why did the coffee not spill during the original turn?
   T2: Will the coffee spill if the mug is nearly full and the robot uses the same turning speed?
   T3: Would using a tray or reducing the turning speed prevent spilling?
   T4: For which fill levels, speeds, and mug types can the robot safely deliver coffee?
   T5: What recorded observations demonstrate that Eva accepted the mug and received the coffee safely?
     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.


The core functional modules can be mapped to the Integration Diagram as follows:
1. Interpret & Ground: Understand Eva's request and identify the intended coffee and an acceptable mug.
2. Perceive: Detect the available mug, its fill level, obstacles, and Eva's current position.
3. Propose: Generate possible actions, such as carrying the mug directly or using a tray.
4. Know & Predict: Use a world model or digital twin to predict possible spills, collisions, and movement feasibility.
5. Decide (Warrant Gate): Check grounding, feasibility, safety, monitoring, repair, and recording criteria before approving an action.
6. Act & Regulate: Execute the approved movement, monitor sensor feedback, and adjust the action if conditions change.
7. Record & Justify: Record the request, assumptions, decisions, observations, and outcome for later verification.

These modules form the Manipulation Executive, which interacts with the Body–World System through continuous perception, action, and feedback.

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.

   The robot should pause and observe Evas new position and replan a safe handover and location and path.
   The potential failure mechanism is a collision with Eva or her walking frame, which could cause the robot to spill hot coffee and injure her. 
   Before committing, the robot must have current sensor evidence confirming that Evas postition is sufficiently known, the revised path is collision-free, the handover location is reachable and the coffee can be carried within the defined stability and safety limits. The robot must also be able to monitor the situation and stop if the conditions would change.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [ ]:
#!/usr/bin/env python3

from collections import Counter
from dataclasses import dataclass
from statistics import pvariance
from typing import Any, Dict, List, Optional, Tuple

import random
import time


@dataclass(frozen=True)
class GoalSpecification:
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float


@dataclass
class ParameterPacket:
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "gaussian_sampler"


class Proposer:
    def __init__(self, seed: Optional[int] = None):
        self.rng = random.Random(seed)

    def propose(self, cautious=False, velocity_override=None, tilt_override=None):
        if cautious:
            mean_velocity, std_velocity = 0.20, 0.025
            mean_tilt, std_tilt = 0.010, 0.003
        else:
            mean_velocity, std_velocity = 0.80, 0.05
            mean_tilt, std_tilt = 0.20, 0.02

        velocity = max(0.01, self.rng.gauss(mean_velocity, std_velocity))
        tilt = max(0.001, self.rng.gauss(mean_tilt, std_tilt))

        if velocity_override is not None:
            velocity = velocity_override
        if tilt_override is not None:
            tilt = tilt_override

        return ParameterPacket(
            grasp_pose=(0.45, 0.12, 0.30),
            max_velocity_mps=round(velocity, 4),
            tilt_rate_radps=round(tilt, 4)
        )


@dataclass
class Observation:
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str


@dataclass
class BeliefState:
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"
    contact_depth_m: Optional[float] = None
    battery_level_pct: float = 80.0
    battery_reserve_pct: float = 5.0


class InterpretationSystem:
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(
            cup_type, {"max_force": 5.0, "max_wave": 2.0}
        )
        busyness_factor = self.kb["room_busyness_limits"].get(
            belief.room_busyness, 0.5
        )
        urgent = "fast" in raw_utterance.lower() or "schnell" in raw_utterance.lower()
        return GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=12.0 if urgent else 15.0,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=1.2 * busyness_factor,
            time_frame_seconds=30.0 if urgent else 60.0
        )


class SensingSystem:
    def sense(self, true_ambient_temp, true_friction, eva_pos, busyness):
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(true_ambient_temp + random.uniform(-0.5, 0.5), 2),
            measured_friction=round(true_friction + random.uniform(-0.05, 0.05), 2),
            eva_position=eva_pos,
            room_busyness=busyness
        )


class PerceptionSystem:
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation):
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification,
                           noise: bool = True) -> Dict[str, float]:
        stiffness = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)

        depth = (self.belief.contact_depth_m if self.belief.contact_depth_m is not None
                 else params.max_velocity_mps * 0.1)
        if depth < 0:
            raise ValueError("Contact depth must not be negative")
        multiplier = 1.8 ** ((depth - 0.02) / 0.01) if depth > 0.02 else 1.0
        force = stiffness * multiplier * params.max_velocity_mps ** 1.5

        degrees_above_20 = max(0.0, self.belief.ambient_temperature_c - 20.0)
        viscosity_factor = 1.0 - 0.01 * degrees_above_20
        if viscosity_factor <= 0:
            raise ValueError("Temperature outside the linear viscosity model")
        viscosity = self.belief.fluid_viscosity_pas * viscosity_factor
        wave_mm = (params.tilt_rate_radps * 0.05 / viscosity) * 1000.0

        force_error = random.uniform(-0.2, 0.3) if noise else 0.0
        wave_error = random.uniform(-0.1, 0.2) if noise else 0.0
        return {
            "predicted_force_n": round(max(0.0, force + force_error), 2),
            "contact_depth_m": round(depth, 4),
            "stiffness_multiplier": round(multiplier, 4),
            "effective_viscosity_pas": round(viscosity, 4),
            "predicted_wave_mm": round(max(0.0, wave_mm + wave_error), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps)
        }


class WarrantGate:
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    @staticmethod
    def estimate_battery_drain_pct(params: ParameterPacket) -> float:
        return 2.0 + 12.0 * params.max_velocity_mps + 4.0 * params.tilt_rate_radps

    def validate(self, params: ParameterPacket, goal: GoalSpecification,
                 belief: BeliefState, noise: bool = True) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal, noise=noise)
        drain = self.estimate_battery_drain_pct(params)
        available = max(0.0, belief.battery_level_pct - belief.battery_reserve_pct)
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0.0 and params.tilt_rate_radps > 0.0,
            "safe": (
                predictions["predicted_force_n"] <= goal.max_allowable_force_n
                and predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm
                and params.max_velocity_mps <= goal.max_nav_velocity_mps
            ),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": drain < 15.0 and drain <= available
        }
        passed = all(checks.values())
        failed = next((name for name, ok in checks.items() if not ok), None)
        next_action = (
            "EXECUTE" if passed else
            "REQUEST_RECHARGE" if not checks["energetically_viable"] else
            "REPLAN_OR_ASK"
        )
        return {
            "passed": passed,
            "failed_criterion": failed,
            "predictions": predictions,
            "warrant_checks": checks,
            "estimated_battery_drain_pct": round(drain, 2),
            "next_action": next_action
        }


class ExecutionEngine:
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification,
                       warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        if not warrant_result["passed"]:
            raise ValueError("Cannot execute a rejected proposal")
        time.sleep(0.01)
        prediction = warrant_result["predictions"]
        actual_force = prediction["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = prediction["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        spill = actual_force > goal.max_allowable_force_n or actual_wave > goal.max_slosh_wave_mm
        return {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml if not spill else max(0.0, goal.target_volume_ml - 3.0),
            "spill_occurred": spill,
            "status": "FAILURE" if spill else "SUCCESS"
        }

    def record_trace(self, asked_utterance: str, goal: GoalSpecification,
                     belief: BeliefState, proposal: ParameterPacket,
                     warrant_result: Dict[str, Any], observation: Observation,
                     committed: bool, execution_outcome: Optional[Dict[str, Any]] = None):
        self.audit_trail.append({
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps
                }
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient,
                "battery_level_pct": belief.battery_level_pct,
                "battery_reserve_pct": belief.battery_reserve_pct
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps
                },
                "warrant_verdict": warrant_result
            },
            "execution_outcome": execution_outcome
        })


def audit_analyzer(audit_trail: List[Dict[str, Any]]) -> Dict[str, Any]:
    rejected = 0
    approved_executions = 0
    failed_counts = Counter()
    errors = []

    for entry in audit_trail:
        verdict = entry["committed"]["warrant_verdict"]
        outcome = entry.get("execution_outcome")
        if not verdict["passed"]:
            rejected += 1
            for criterion, ok in verdict["warrant_checks"].items():
                if not ok:
                    failed_counts[criterion] += 1
        elif entry["committed"]["action_approved"] and outcome is not None:
            approved_executions += 1
            predicted = verdict["predictions"]["predicted_force_n"]
            errors.append(outcome["actual_force_n"] - predicted)

    if failed_counts:
        maximum = max(failed_counts.values())
        most_frequent = sorted(k for k, v in failed_counts.items() if v == maximum)
    else:
        most_frequent = []

    variance = pvariance(errors) if len(errors) >= 2 else None
    result = {
        "rejected_proposals": rejected,
        "approved_executions": approved_executions,
        "failure_counts": dict(failed_counts),
        "most_frequent_criteria": most_frequent,
        "force_error_variance_n2": variance
    }
    print("\nAudit analysis")
    for name, value in result.items():
        print(f"{name}: {value}")
    return result


KNOWLEDGE_BASE = {
    "cups": {
        "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
        "plastic_cup": {"max_force": 7.0, "max_wave": 3.0}
    },
    "room_busyness_limits": {"low": 1.0, "medium": 0.6, "high": 0.3}
}


def run_episode(belief: BeliefState, params: ParameterPacket,
                command: str = "Bring me my coffee", noise: bool = False,
                force_error: Optional[float] = None):
    observation = Observation(
        time.time(), belief.ambient_temperature_c,
        belief.surface_friction_coefficient, belief.eva_position, belief.room_busyness
    )
    goal = InterpretationSystem(KNOWLEDGE_BASE).interpret(command, belief)
    gate = WarrantGate(SemanticDigitalTwin(belief))
    executive = ExecutionEngine()
    verdict = gate.validate(params, goal, belief, noise=noise)
    outcome = None
    if verdict["passed"]:
        if force_error is None:
            outcome = executive.execute_action(params, goal, verdict)
        else:
            # Controlled physical observation for the audit test
            predicted_force = verdict["predictions"]["predicted_force_n"]
            outcome = {
                "actual_force_n": round(predicted_force + force_error, 2),
                "actual_wave_mm": verdict["predictions"]["predicted_wave_mm"],
                "actual_volume_transferred_ml": goal.target_volume_ml,
                "spill_occurred": False,
                "status": "SUCCESS"
            }
    executive.record_trace(command, goal, belief, params, verdict, observation,
                           committed=verdict["passed"], execution_outcome=outcome)
    return verdict, executive.audit_trail[0]


def test_contact():
    proposer = Proposer(seed=42)
    params = proposer.propose(velocity_override=0.35, tilt_override=0.01)
    verdict, _ = run_episode(BeliefState(), params)
    pred = verdict["predictions"]
    assert pred["contact_depth_m"] == 0.035
    assert abs(pred["predicted_force_n"] - 6.53) < 0.01
    assert not verdict["passed"] and verdict["failed_criterion"] == "safe"
    assert verdict["next_action"] == "REPLAN_OR_ASK"
    print("Contact test:", pred["predicted_force_n"], "N,", verdict["next_action"])


def test_temperature():
    proposer = Proposer(seed=42)
    params = proposer.propose(velocity_override=0.10, tilt_override=0.15)
    at_20, _ = run_episode(BeliefState(ambient_temperature_c=20.0), params)
    at_28, _ = run_episode(BeliefState(ambient_temperature_c=28.0), params)
    pred_20 = at_20["predictions"]
    pred_28 = at_28["predictions"]
    assert pred_20["predicted_wave_mm"] == 6.25
    assert pred_28["predicted_wave_mm"] == 6.79
    assert not at_28["warrant_checks"]["safe"]
    print("Temperature test:", pred_20["predicted_wave_mm"], "->",
          pred_28["predicted_wave_mm"], "mm,", at_28["next_action"])


def test_low_battery():
    params = Proposer(seed=42).propose(velocity_override=0.10, tilt_override=0.005)
    verdict, record = run_episode(BeliefState(battery_level_pct=6.0), params)
    assert verdict["estimated_battery_drain_pct"] == 3.22
    assert not verdict["warrant_checks"]["energetically_viable"]
    assert all(ok for name, ok in verdict["warrant_checks"].items()
               if name != "energetically_viable")
    assert verdict["next_action"] == "REQUEST_RECHARGE"
    assert record["execution_outcome"] is None
    print("Battery test:", verdict["estimated_battery_drain_pct"], "%,",
          verdict["next_action"])


def test_audit():
    proposer = Proposer(seed=42)
    rejected_contact = proposer.propose(velocity_override=0.35, tilt_override=0.01)
    low_battery = proposer.propose(velocity_override=0.10, tilt_override=0.005)
    safe_params = proposer.propose(velocity_override=0.10, tilt_override=0.005)

    entries = [
        run_episode(BeliefState(), rejected_contact)[1],
        run_episode(BeliefState(battery_level_pct=6.0), low_battery)[1],
        run_episode(BeliefState(), safe_params, force_error=0.10)[1],
        run_episode(BeliefState(), safe_params, force_error=0.30)[1]
    ]
    result = audit_analyzer(entries)
    assert result["rejected_proposals"] == 2
    assert result["approved_executions"] == 2
    assert result["most_frequent_criteria"] == ["energetically_viable", "safe"]
    assert abs(result["force_error_variance_n2"] - 0.01) < 1e-10


def main():
    random.seed(42)
    belief = BeliefState()
    sensing = SensingSystem()
    perception = PerceptionSystem(belief)
    observation = sensing.sense(21.5, 0.45, "living_room", "medium")
    perception.update(observation)
    print("Observation:", observation)
    print("Belief:", belief)

    proposer = Proposer(seed=42)
    goal = InterpretationSystem(KNOWLEDGE_BASE).interpret("Bring me my coffee fast!", belief)
    gate = WarrantGate(SemanticDigitalTwin(belief))
    executive = ExecutionEngine()

    for attempt in range(2):
        params = proposer.propose(cautious=attempt > 0)
        verdict = gate.validate(params, goal, belief)
        outcome = (executive.execute_action(params, goal, verdict)
                   if verdict["passed"] else None)
        executive.record_trace("Bring me my coffee fast!", goal, belief, params,
                               verdict, observation, verdict["passed"], outcome)
        print("Attempt", attempt + 1, verdict["next_action"], verdict["predictions"])
        if verdict["passed"]:
            print("Outcome:", outcome)
            break

    print("\nTests")
    test_contact()
    test_temperature()
    test_low_battery()
    test_audit()
    print("All tests passed")


if __name__ == "__main__":
    main()


## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.

The classes correspond to the functional  blocks of the lectures Integration Diagram.
InterpretationSystem.interpret() translates Eva's natural-language request to a concrete GoalSpecification.SensingSystem.sense() generates raw observations, when PerceptionSystem.update() updates the internal BeliefState.

Proposer.propose() represents System 1 and generates candidate movement parameters. SemanticDigitalTwin.predict_properties() represents the predictive world model and estimates the pysical consequences of these parameters.

WarrantGate.validate() checks wheter the proposed action is satisfies the grounding, feasibility, safety, monitoring, repair and recording criteria.

If a proposal is approved, ExecutionEngine.execute_action() simulates the physical execution. ExecutionEngine.record_trace() records the request, beliefs, observations, decisions and outgoings.

During one cognitive decision cycle, the sensor data update the belief state, the request is interpreted, an action is proposed, its conseuences are predicted and the Warrant Gate decides whether execution is justified. Rejected proposals can be revised and checked again. This forms a feedback-based congnitive architecture.

* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

I implemented the Propser class using a Gaussian probability distribution to enerate candidate movement poarameters.
I choose the Gaussian distribution because it generates values predefined mean values, while extreme deviations are less likely.
The Proposer generates the robots maximumum velocity and tilt rate using random.guess(). It supports two operating modes, a normal mode and a cautious mode with lower mean velocities and tilt rates.
When the Warrant Gate rejects an inital proposal, the Proposer could generate a more conservate alternative. This demonstrates the seperation beween System1, which generates possible actions and System 2, which predicts their consequences and checks whether execution is justified.

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

I modified the SemanticDigitalTwin.predict_properties() method to implement a dual-stage material compliance model.

For contact dephts up to 0,02m the original stiffness coefficient remains unchanged. Beyond this threshold, the stiffness increases exponentially by a factor of 1,8 for every additional 0,01 m of penetration. This models the flexible pouch bottoming out a rigid internal fixture.

Since the original code does not provide a contact-depth measurement, I assumend an approach duration of 0,1 seconds to estimate the contact depth from the approach velocity. The characteristc penetration distance of 0,01 m is an additonal modeling assumption.

* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

I implemented a test case in which the Proposer generates an aggressive approach velocity of 0,35 m/s.

Using the assumed contact duration of 0,1 seconds, the resulting contact depth is 0,035 m, exceeding the threshold of 0,02 m. Therefore, the exponential stiffness multiplier becomes approximately 2,415.

The semantic digital twin predicts a contact force of approximately 6,53 N exceeding the allowable limits of 4,5 N.

Consequently, the Warrant Gate rejects the parameter packet because the safe criterion fails.

An assertion verifies the rejection, and a mocked execution method confirms that physical execution is never initiated. The test therefore demonstrates that an unsafe proposal is intercepted before execution.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).

I enhanced the SemanticDigitalTwin.predict_properties() method by introducing a linear temprature-dependent viscosity model.

The effective viscosity decreases by 1% for every degree Celsius above the reference temperature of 20°C.

The model uses the equation: 

n(T) = n0 * [1 -0,01 * (T - 20)]

for temperature above 20°C.

For example at 28°C, the viscosity decreases from 1,2 tto 1,104.

The effective viscosity decreases is then used to calculate the predicted wave height. Lower viscosity results in a larger predicted wave height for the same tilt in this simplified model.


* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

I simulated the fluid transport at ambient temperatures of 20°C and 28°C with a fixed filt rate of 0,15 rad/s.

At 20°C, the effective viscosity is 1,2 resulating in a predicted wave height of 6,25 mm.

At 28°C, the viscosity decreases to 1,1014 increasing the predicted wave height to approximately 6,79 mm.

This corresponds to an increase of approximately 8,7%.

The maximum allowable wave height is 1,5 mm. Therefore, the Warrant Gate rejects the proposal because the safe criterion is violated.

The safety criterion is already violated at 20°C, but the higher temperature further increases the predicated wave height.

The simulation demonstrates how temperature-dependen fluid properties can influence action validation in an embodied cognitive archtiecture.

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.

I extended the WarrantGate with a seventh criterion called energetically_visable. The criterion evaluates whether the estimated energy consumption of a proposed trajectory remains below 15% of the total battery capacity.

Since the assignment does not specify an energy consumption model, I implemented a simplified estimation based on movement velocity and tilt rate.

Additionally, the check verifies that enough battery charge remains to execute the trajectory while maintaining a minimum battery reserve.

A proposal is approved only if all seven warrant criteria are satisfied.

* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

I extended the BeliefState with battery_level_pct and battery_reserver_pctto represent the robots current battery charge and required reserve.

In the test scenario, the robot has 6% battery remaining and must maintain a minimum reserve of 5%. The proposed coffee delivery requires an estimted 3,22% of total battery capacity. Athough this is below 15% trajectory consumuption limit, completing the movement would leave only 2,78% battery vioalating the rserver requirement.

Therefore, the energetically_vialbe check fails and the Warrant Gate rejects the proposal.

The system redirects its next action to REQUEST_RECHARGE instead of executing the coffee delivery. The test confirms that physical execution is blocked when the energy requirements are not satisfied.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.

I implemented the audit_analyzer(audit_trail) function to evaluate the robots recorded episodic decisions.

The function counts rejected proposals and approved executions, identifies the warrant criteria most frequently responsible for rejection and calcualtes the population variance of the force pediction errors.

For the test, four episodes were recorded, so two rejected proposals and two approved executions.

The rejected proposals failed the safe and energetically_vialbe criteria, respectivaley. Therefore both criteria were eually frequent causes of rejection.

For the two executed episodes controlled force prediction errors of 0,10 N and 0,30 N were used. The calculated population variance was 0,0100 N^2 

Only executed episodes with available predicted and observed force values were included in the variance calculation.

This demonstrates how an episodic audit trail supports systematic evaluation of robot decisions and prediction accuracy.

* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

Maintaing seperate records for asked, belief, committed and observed makes the robots decision-making process transparent and auditable.

The asked record represent the users original request and intended goal. The belief record describes the robots internal representation of the environment.
The committed record documents the selected action its predicted consequences and Warrant Gate decision. The observerd record contains the actual sensor measurements.

Additonally the execution_outcome records the physical result of an executed action. Pure imitation learning primarily learns action patterns from demonstrations. Although learned policies may contain predictive knowledge successful imitation alone does not establish why an action was appropriate whether its assumptions were valid or under which changed conditionbs it would fail.

By seperating the requested goal, internal beliefs, action commitments and observations, the architecture enables the comparison of predictions with actual outcomes.

For example if the robot spills coffee, the audit trail can help determine whether the failure resulted from an incorrect environmental belief, an inaccurate physical prediction, or an insufficient safety decision.

This separation therefore supports causal investigation, error detection, systematic improvement and evidence-based competence claims.

However maintating these records does not automatically fuarantee correct explanations or safe behavior. The recorded information must still be accurate, complete and evaluated against clearly defined criteria.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

---


The robot consists of several functional components that enable perception, movement, manipulation and interaction with its environment.

Links are the rigid mechanical structures connecting the robots joints such as the upper and lower arm segments.

Joints connect these links and allow relative motion between them. Actuators such as electric motors etc. generate the forces and movements required to control the joints.

Sensors collect information about the robot and its surroundings. Cameras and distance sensors help to identify objects, estimate their positions and detect obastacles.

Effectors, particulary the robots grippers, allow it to manipulate objects such as cups.

The base supports the robot and provides mobility through its wheels. The torso connects the mobile base to the upper body, while the head can carry sensors for environmental perception. Unlike legged robots, the illustrated robot uses wheels for loccomotion.


The robot also uses different forms of perception:

- Proprioception: Information about its own movements and joint positions.
- Exteroception: Information about external objects and obstacles through cameras and distance sensors
- Interoception Information about internal operating conditions, such as battery level and motor temperature.

These components support the emboidied cognitive architecture from Part 2. Sensors provide observations to the Perception System, which updates the BeliefState. The Semantic Digital Twin predicts the physical consequences of proposed movements. The Warrant Gate checks whether an action is justified, while the Execution Engine controls the actuators and effectors.

Together, theses components allow the robot to perceive its environment, plan movements, manipulate objects and monitor execution.

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 